In [1]:
# Import packages and setup Neo4j for cypher query language.


from dotenv import load_dotenv
import os 

from langchain_community.graphs import Neo4jGraph

In [9]:
import warnings
warnings.filterwarnings("ignore")

Need to setup the database in neo4j, by creating instance and creating database.
get the environment variables and set it in you environment and then run the following query.

In [2]:
load_dotenv()

# os.environ["NEO4J_URI"] = os.getenv("NEO4J_URI")
# os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME")
# os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD")
# os.environ["NEO4J_DATABASE"] = os.getenv("NEO4J_DATABASE")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [3]:
print(NEO4J_URI)

neo4j+s://1538dc1c.databases.neo4j.io


In [4]:
kg = Neo4jGraph(
    url=NEO4J_URI, 
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD, 
    database=NEO4J_DATABASE
)

/var/folders/s8/qyjb36g92fs3ztdqk120mmkw0000gn/T/ipykernel_4995/1012656988.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(


(person)-[Acted_IN]->(Movie)

person -> we know person acted in some movies so actor becomes actor because they have acted in something.


(person) :- Properties (name:string, born:integer)
(movie) :- properties (title: string, tagline:string, released: integer)

We can see the relationships:-

              |---[ACTED_IN]-->|
              |---[DIRECTED]-->|
              |---[WROTE]----->|
|-->(person)--|---[PRODUCED]-->|--(Movie)
|<-----|      |---[REVIEWED]-->|
[FOLLOWS]


In [ ]:
# Querying the movie Knowledge graph
# Cypher - > Neo4j's query language it is using pattern matching to find things inside the grass.
#  Cypher begins with MATCH clause pattern matching smallest pattern matching it looks for is the single node pattern.

cypher = """ 
MATCH (n)                  
RETURN count(n)
"""


In [6]:
result = kg.query(cypher)
result

[#C691]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.114.186', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 1.151115248992339s (Unable to retrieve routing information)
[#C693]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-1538dc1c-8c54.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): OSError('No data')
Transaction failed and will be retried in 1.7143894546301164s (Failed to read from defunct connection IPv4Address(('si-1538dc1c-8c54.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))


[{'count(n)': 171}]

result is actually a list of rows basically.It comes backs from the query kg, and in that query, each of the rows is then a dictionary where dictionary keys

are based on what you have returned from the return clause up here in the cipher.

In [10]:
# naming count as alias

cypher = """ 
MATCH (n)
RETURN count (n) AS numberOfNodes
"""

In [11]:
result = kg.query(cypher)
result

[{'numberOfNodes': 171}]

In [12]:
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 171 nodes in this graph.


In [17]:
# We just want to finds nodes with movies and persons.

cypher = """ 
MATCH (m:Movie)
RETURN count(m) AS numberOfMovies
"""

In [18]:
result = kg.query(cypher)
result

[{'numberOfMovies': 38}]

In [20]:
# nodes with person

cypher = """ 
MATCH (p:Person)
RETURN count(p) AS numberOfPeople
"""
kg.query(cypher)

[{'numberOfPeople': 133}]

In [27]:
# lets get person name tom hanks

cypher = """ 
MATCH (tom:Person {name:"Tom Cruise"})
RETURN tom
"""

kg.query(cypher)

[{'tom': {'born': 1962, 'name': 'Tom Cruise'}}]

In [29]:
# lets look for movie
cypher = """ 
MATCH (cloudAtlas:Movie {title:"Cloud Atlas"})
RETURN cloudAtlas
"""

kg.query(cypher)

[{'cloudAtlas': {'tagline': 'Everything is connected',
   'title': 'Cloud Atlas',
   'released': 2012}}]

In [32]:
# we just want release data of the movie

cypher = """ 
MATCH (cloudAtlas:Movie {title:"Cloud Atlas"})
RETURN cloudAtlas.released
"""

kg.query(cypher)

[{'cloudAtlas.released': 2012}]

In [33]:
# will return 2 things in addition to released date

cypher =""" 
MATCH (cloudAtlas:Movie {title:"Cloud Atlas"})
RETURN cloudAtlas.released, cloudAtlas.tagline
"""

kg.query(cypher)

[#C7A3]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.114.186', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 1.135061657731418s (Unable to retrieve routing information)
[#C751]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-1538dc1c-8c54.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): OSError('No data')
Transaction failed and will be retried in 2.3200720410249707s (Failed to read from defunct connection IPv4Address(('si-1538dc1c-8c54.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))


[{'cloudAtlas.released': 2012,
  'cloudAtlas.tagline': 'Everything is connected'}]

In [34]:
# Cypher patterns with conditional matching
# return all the movies from 1990s

cypher = """ 
MATCH (nineties:Movie)
WHERE nineties.released >= 1990
  AND nineties.released < 2000
RETURN nineties.title
"""

kg.query(cypher)


[{'nineties.title': 'The Matrix'},
 {'nineties.title': "The Devil's Advocate"},
 {'nineties.title': 'A Few Good Men'},
 {'nineties.title': 'As Good as It Gets'},
 {'nineties.title': 'What Dreams May Come'},
 {'nineties.title': 'Snow Falling on Cedars'},
 {'nineties.title': "You've Got Mail"},
 {'nineties.title': 'Sleepless in Seattle'},
 {'nineties.title': 'Joe Versus the Volcano'},
 {'nineties.title': 'When Harry Met Sally'},
 {'nineties.title': 'That Thing You Do'},
 {'nineties.title': 'The Birdcage'},
 {'nineties.title': 'Unforgiven'},
 {'nineties.title': 'Johnny Mnemonic'},
 {'nineties.title': 'The Green Mile'},
 {'nineties.title': 'Hoffa'},
 {'nineties.title': 'Apollo 13'},
 {'nineties.title': 'Twister'},
 {'nineties.title': 'Bicentennial Man'},
 {'nineties.title': 'A League of Their Own'}]

In [37]:
# pattern matching with multiples nodes
# people who acted in movies names of people and they acted in the movie

cypher = """ 
MATCH (actor:Person)-[:ACTED_IN]->(movie:Movie)
RETURN actor.name AS Actor, movie.title AS Movie LIMIT 10
"""

kg.query(cypher)



[{'Actor': 'Keanu Reeves', 'Movie': 'The Matrix'},
 {'Actor': 'Carrie-Anne Moss', 'Movie': 'The Matrix'},
 {'Actor': 'Laurence Fishburne', 'Movie': 'The Matrix'},
 {'Actor': 'Hugo Weaving', 'Movie': 'The Matrix'},
 {'Actor': 'Emil Eifrem', 'Movie': 'The Matrix'},
 {'Actor': 'Keanu Reeves', 'Movie': 'The Matrix Reloaded'},
 {'Actor': 'Carrie-Anne Moss', 'Movie': 'The Matrix Reloaded'},
 {'Actor': 'Laurence Fishburne', 'Movie': 'The Matrix Reloaded'},
 {'Actor': 'Hugo Weaving', 'Movie': 'The Matrix Reloaded'},
 {'Actor': 'Keanu Reeves', 'Movie': 'The Matrix Revolutions'}]

Here we can see Emil Eifrem is not a actor nor he acted in Matrix movie still we can see him.

In [40]:
# lets find out movies in which tom hanks had worked. 

cypher = """ 
MATCH (tom:Person {name:"Tom Hanks"})-[:ACTED_IN]->(tomHanksMovie:Movie)
RETURN tom.name, tomHanksMovie.title
"""

kg.query(cypher)

[{'tom.name': 'Tom Hanks', 'tomHanksMovie.title': "You've Got Mail"},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'Sleepless in Seattle'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'Joe Versus the Volcano'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'That Thing You Do'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'Cloud Atlas'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'The Da Vinci Code'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'The Green Mile'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'Apollo 13'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'Cast Away'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': "Charlie Wilson's War"},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'The Polar Express'},
 {'tom.name': 'Tom Hanks', 'tomHanksMovie.title': 'A League of Their Own'}]

In [41]:
# person acted in movie to some other person who acted in the same movie

cypher = """ 
MATCH (tom:Person {name:"Tom Hanks"})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors)
RETURN coActors.name, m.title
"""

kg.query(cypher)

[{'coActors.name': 'Meg Ryan', 'm.title': "You've Got Mail"},
 {'coActors.name': 'Greg Kinnear', 'm.title': "You've Got Mail"},
 {'coActors.name': 'Parker Posey', 'm.title': "You've Got Mail"},
 {'coActors.name': 'Dave Chappelle', 'm.title': "You've Got Mail"},
 {'coActors.name': 'Steve Zahn', 'm.title': "You've Got Mail"},
 {'coActors.name': 'Meg Ryan', 'm.title': 'Sleepless in Seattle'},
 {'coActors.name': 'Rita Wilson', 'm.title': 'Sleepless in Seattle'},
 {'coActors.name': 'Bill Pullman', 'm.title': 'Sleepless in Seattle'},
 {'coActors.name': 'Victor Garber', 'm.title': 'Sleepless in Seattle'},
 {'coActors.name': "Rosie O'Donnell", 'm.title': 'Sleepless in Seattle'},
 {'coActors.name': 'Meg Ryan', 'm.title': 'Joe Versus the Volcano'},
 {'coActors.name': 'Nathan Lane', 'm.title': 'Joe Versus the Volcano'},
 {'coActors.name': 'Charlize Theron', 'm.title': 'That Thing You Do'},
 {'coActors.name': 'Liv Tyler', 'm.title': 'That Thing You Do'},
 {'coActors.name': 'Hugo Weaving', 'm.title

In [48]:
# Delete data from the graph

cypher = """ 
MATCH (emil:Person {name:"Emil Eifrem"})-[actedIn:ACTED_IN]->(movie:Movie)
RETURN emil.name, movie.title
"""

kg.query(cypher)

[]

In [46]:
# wil  delete acted in relationship for the above query. with DELETE keyword and whatever relationship you want

cypher = """ 
MATCH (emil:Person {name:"Emil Eifrem"})-[actedIn:ACTED_IN]->(movie:Movie)
DELETE actedIn
"""

kg.query(cypher)


[]

In [50]:
# Creating data similar to doing matching omn single nodes.
# Adding data to the graph

cypher = """ 
CREATE (utkarsh:Person {name:"Utkarsh"})
RETURN utkarsh
"""

kg.query(cypher)

[{'utkarsh': {'name': 'Utkarsh'}}]

In [51]:
# Lets add relationship with emil founder of neo4j an utkarsh

cypher = """ 
MATCH (utkarsh:Person {name:"Utkarsh"}),(emil:Person {name:"Emil Eifrem"})
MERGE (utkarsh)-[hasRelationship:WORKS_WITH]->(emil)
RETURN utkarsh, hasRelationship, emil
"""

kg.query(cypher)

[{'utkarsh': {'name': 'Utkarsh'},
  'hasRelationship': ({'name': 'Utkarsh'},
   'WORKS_WITH',
   {'born': 1978, 'name': 'Emil Eifrem'}),
  'emil': {'born': 1978, 'name': 'Emil Eifrem'}}]